In [ ]:
import threading
import time
import queue
import random

KURSI_TUNGGU = 3
JUMLAH_PELANGGAN = 7

# Queue berfungsi ganda: menyimpan ID pelanggan sekaligus mengatur 
# giliran otomatis bagi 2 tukang cukur yang sedang idle/kosong
ruang_tunggu = queue.Queue(maxsize=KURSI_TUNGGU)
print_lock = threading.Lock()

def log(pesan):
    with print_lock:
        print(pesan)

def tukang_cukur(id_barber):
    while True:
        # Tukang cukur tidur/menunggu sampai ada pelanggan di queue
        pelanggan = ruang_tunggu.get()
        
        # Sinyal None menandakan toko tutup (semua pelanggan selesai)
        if pelanggan is None:
            log(f"[Barber {id_barber}] Toko tutup, pulang.")
            ruang_tunggu.task_done()
            break
            
        log(f"  --> [Barber {id_barber}] Mencukur Pelanggan {pelanggan} "
            f"(Sisa antrean: {ruang_tunggu.qsize()}/{KURSI_TUNGGU})")
        time.sleep(random.uniform(0.1, 0.2)) # Durasi mencukur
        log(f"  [V] [Barber {id_barber}] Selesai mencukur Pelanggan {pelanggan}")
        ruang_tunggu.task_done()

def proses_pelanggan(id_pelanggan):
    try:
        # Coba masuk ruang tunggu tanpa blocking (put_nowait)
        ruang_tunggu.put_nowait(id_pelanggan)
        log(f"[Pelanggan {id_pelanggan}] Duduk di ruang tunggu "
            f"(Terisi: {ruang_tunggu.qsize()}/{KURSI_TUNGGU})")
    except queue.Full:
        log(f"[X] [Pelanggan {id_pelanggan}] Ruang tunggu penuh! Langsung pulang.")


barbers = [
    threading.Thread(target=tukang_cukur, args=(1,)),
    threading.Thread(target=tukang_cukur, args=(2,))
]
for b in barbers:
    b.start()


for i in range(1, JUMLAH_PELANGGAN + 1):
    proses_pelanggan(i)
    time.sleep(0.04)


ruang_tunggu.join()

for _ in barbers:
    ruang_tunggu.put(None)
for b in barbers:
    b.join()

print("\nSimulasi 2 Tukang Cukur selesai!")

[Pelanggan 1] Duduk di ruang tunggu (Terisi: 1/3)
  --> [Barber 1] Mencukur Pelanggan 1 (Sisa antrean: 0/3)
[Pelanggan 2] Duduk di ruang tunggu (Terisi: 1/3)
  --> [Barber 2] Mencukur Pelanggan 2 (Sisa antrean: 0/3)
[Pelanggan 3] Duduk di ruang tunggu (Terisi: 1/3)
[Pelanggan 4] Duduk di ruang tunggu (Terisi: 2/3)
  [V] [Barber 2] Selesai mencukur Pelanggan 2
  --> [Barber 2] Mencukur Pelanggan 3 (Sisa antrean: 2/3)
[Pelanggan 5] Duduk di ruang tunggu (Terisi: 3/3)
  [V] [Barber 1] Selesai mencukur Pelanggan 1
  --> [Barber 1] Mencukur Pelanggan 4 (Sisa antrean: 1/3)
[Pelanggan 6] Duduk di ruang tunggu (Terisi: 2/3)
[Pelanggan 7] Duduk di ruang tunggu (Terisi: 3/3)
  [V] [Barber 1] Selesai mencukur Pelanggan 4
  --> [Barber 1] Mencukur Pelanggan 5 (Sisa antrean: 2/3)
  [V] [Barber 2] Selesai mencukur Pelanggan 3
  --> [Barber 2] Mencukur Pelanggan 6 (Sisa antrean: 1/3)
  [V] [Barber 1] Selesai mencukur Pelanggan 5
  --> [Barber 1] Mencukur Pelanggan 7 (Sisa antrean: 0/3)
  [V] [Barber 

In [ ]:
Untuk membagikan pelanggan ke 2 tukang cukur yang berbagi satu ruang 
tunggu yang sama, digunakan `queue.Queue(maxsize=3)` yang sudah 
bersifat thread-safe.

Cara kerjanya:
1. Jika kedua tukang cukur sedang menganggur (tidur), keduanya sama-sama 
   memanggil `ruang_tunggu.get()`.
2. Begitu satu pelanggan masuk lewat `put_nowait()`, mekanisme internal 
   Queue hanya akan membangunkan satu tukang cukur yang paling cepat 
   mengambil antrean tersebut, sementara tukang cukur satunya tetap 
   menunggu pelanggan berikutnya.
3. Jika ke-3 kursi tunggu penuh (`queue.Full`), pelanggan baru otomatis 
   meninggalkan toko tanpa membuat program macet.